In [1]:
#=====================
#STEP 0: Mount DATA
#=====================

data_dir = r"C:\Tutorial\Visuara\Computer vision from Scratch\flowers"

In [2]:
import sys
print(sys.executable)



C:\PYTHON_ENV\env_object_detection\Scripts\python.exe


In [3]:
#pip install --upgrade wandb

In [4]:
# =======================
# STEP 1: Imports
# =======================
import tensorflow as tf
from tensorflow.keras.applications import NASNetLarge
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.optimizers import Adam
import wandb
#from wandb.keras import WandbMetricsLogger, WandbModelCheckpoint
from wandb.integration.keras import WandbMetricsLogger, WandbModelCheckpoint

In [5]:
# =======================
# STEP 2: Initialize Weights & Biases
# =======================
wandb.init(project="NASNetLarge-flowers", config={
    "epochs": 5,
    "batch_size": 16,
    "learning_rate": 0.001,
    "architecture": "NASNetLarge",
    "pretrained": True,
    "input_size": 331
})
config = wandb.config


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\sangr\_netrc.
wandb: Currently logged in as: sangram01 (sangram01-srm-institute-of-science-and-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
# =======================
# STEP 3: Data Preparation
# =======================
IMAGE_SIZE = (331, 331)  # Required input size for NASNetLarge

train_dir = r"C:\Tutorial\Visuara\Computer vision from Scratch\flowers/train"
val_dir = r"C:\Tutorial\Visuara\Computer vision from Scratch\flowers/val"

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SIZE,
    batch_size=config.batch_size,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMAGE_SIZE,
    batch_size=config.batch_size,
    class_mode='categorical'
)

Found 4008 images belonging to 5 classes.
Found 1011 images belonging to 5 classes.


In [7]:
# =======================
# STEP 4: Load NASNetLarge Model
# =======================
base_model = NASNetLarge(weights='imagenet', include_top=False, input_shape=(331, 331, 3))

# Freeze all convolutional layers
for layer in base_model.layers:
    layer.trainable = False

# Add custom classifier
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(1024, activation='relu')(x)
predictions = Dense(5, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

In [8]:
# =======================
# STEP 5: Compile & Train
# =======================
model.compile(optimizer=Adam(learning_rate=config.learning_rate),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

callbacks = [
    WandbMetricsLogger(log_freq="epoch"),
    WandbModelCheckpoint(filepath="model.keras", monitor="val_accuracy", save_best_only=True)
]

model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=config.epochs,
    callbacks=callbacks
)

wandb: WARNING When using `save_best_only`, ensure that the `filepath` argument contains formatting placeholders like `{epoch:02d}` or `{batch:02d}`. This ensures correct interpretation of the logged artifacts.


Epoch 1/5
  1/251 ━━━━━━━━━━━━━━━━━━━━ 2:54:41 42s/step - accuracy: 0.2500 - loss: 1.6294

KeyboardInterrupt: 